# CivicDesk 311 Ticket Rewriter — LoRA SFT + DPO on a free-tier Colab GPU

Adapts Qwen2.5-0.5B-Instruct with LoRA to rewrite messy resident 311 complaints into structured dispatch tickets, then runs a short DPO pass on top of the same adapter.

**Before running:** Runtime > Change runtime type > T4 GPU (free tier).

Every training run below writes `run_metrics.json` (peak GPU memory + wall-clock time) next to its checkpoint, and the evaluation step records generation wall-clock time — this is the evidence that the whole pipeline fits the free-tier hardware budget.

In [1]:
!nvidia-smi

Fri Sep 25 23:50:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone the repo

In [2]:
!git clone https://github.com/nmilleson/csp-cst627-week3.git
%cd csp-cst627-week3

Cloning into 'csp-cst627-week3'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 33 (delta 8), reused 33 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 50.44 KiB | 8.41 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/csp-cst627-week3


## 2. Install dependencies

Colab GPU runtimes ship a CUDA-matched `torch` already, so this just adds the rest on top.

In [3]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00


In [4]:
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

torch 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
total memory (GB): 15.637086208


## 3. Dataset

`data/train.jsonl`, `data/val.jsonl`, `data/dpo_train.jsonl`, and `data/dpo_val.jsonl` are already committed to the repo. This cell regenerates them from scratch with the same seeds, just to show the pipeline is fully reproducible end to end.

In [5]:
!python data/generate_dataset.py --n-per-category 30 --seed 7 --output-dir data
!python data/generate_preferences.py --input data/train.jsonl --output data/dpo_train.jsonl --seed 11
!python data/generate_preferences.py --input data/val.jsonl --output data/dpo_val.jsonl --seed 12

wrote 300 examples to data/all.jsonl
wrote 270 examples to data/train.jsonl
wrote 30 examples to data/val.jsonl
wrote 270 preference pairs to data/dpo_train.jsonl
wrote 30 preference pairs to data/dpo_val.jsonl


## 4. LoRA SFT pass

QLoRA (4-bit) fine-tune of Qwen2.5-0.5B-Instruct on the 270 training pairs. LoRA rank 16 keeps the trainable adapter to a few million parameters.

In [6]:
!python training/sft_train.py \
    --train data/train.jsonl --val data/val.jsonl \
    --output-dir checkpoints/sft-adapter

config.json: 100% 659/659 [00:00<00:00, 1.72MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 3.75MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 71.8MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 80.4MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 103MB/s]

model.safetensors: downloading bytes:   1% 12.0M/988M [00:00<00:50, 19.2MB/s]
model.safetensors: downloading bytes:  15% 150M/988M [00:01<00:03, 223MB/s, 11.3MB/s  ]
model.safetensors: reconstructing file:   8% 83.9M/988M [00:01<00:10, 86.6MB/s, 1.65MB/s  ]
model.safetensors: downloading bytes:  20% 194M/988M [00:01<00:03, 221MB/s, 16.3MB/s  ]
model.safetensors: downloading bytes:  48% 471M/988M [00:02<00:01, 282MB/s, 41.2MB/s  ]
model.safetensors: downloading bytes:  56% 556M/988M [00:02<00:01, 235MB/s, 47.3MB/s  ]
model.safetensors: downloading bytes:  86% 845M/988M [00:03<00:00, 305MB/s, 67.3MB/s  ]
model.safetensors: reconstructing file:  54% 537M/988M [00:04<00:04, 107MB/s, 35.9MB/s  ]
model.safetensors: down

In [7]:
import json
print(json.dumps(json.load(open("checkpoints/sft-adapter/run_metrics.json")), indent=2))

{
  "wall_clock_seconds": 286.81222677230835,
  "peak_gpu_memory_gb": 1.213894144
}


## 5. DPO pass on top of the SFT adapter

Continues training the *same* adapter with preference pairs (chosen vs. rejected tickets), using the frozen base model (adapter disabled) as the implicit reference policy — no second full model copy needed.

In [8]:
!python training/dpo_train.py \
    --sft-adapter checkpoints/sft-adapter \
    --train data/dpo_train.jsonl --val data/dpo_val.jsonl \
    --output-dir checkpoints/dpo-adapter

Loading weights: 100% 290/290 [00:00<00:00, 510.76it/s]
Generating train split: 270 examples [00:00, 78187.11 examples/s]
Generating train split: 30 examples [00:00, 13421.77 examples/s]
Map: 100% 270/270 [00:00<00:00, 12546.81 examples/s]
Map: 100% 30/30 [00:00<00:00, 4555.89 examples/s]
Tokenizing train dataset: 100% 270/270 [00:00<00:00, 547.73 examples/s]
Dropping fully truncated examples from train dataset: 100% 270/270 [00:00<00:00, 6509.90 examples/s]
Tokenizing eval dataset: 100% 30/30 [00:00<00:00, 510.13 examples/s]
Dropping fully truncated examples from eval dataset: 100% 30/30 [00:00<00:00, 3730.81 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
{'loss': '0.6601', 'grad_norm': '15', 'learning_rate': '4.5e-06', 'entropy': '0.1174', 'num_tokens': '2.687e+

In [9]:
print(json.dumps(json.load(open("checkpoints/dpo-adapter/run_metrics.json")), indent=2))

{
  "wall_clock_seconds": 241.27807474136353,
  "peak_gpu_memory_gb": 5.865288192
}


## 6. Evaluate base vs. SFT vs. DPO

Same 30 held-out complaints, greedy decoding, scored for JSON validity, schema completeness, and category/priority accuracy against the ground-truth ticket.

In [14]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.1 MB/s eta 0:00:00


In [15]:
!python training/evaluate.py --label base --eval-file data/val.jsonl --output eval_results/base
!python training/evaluate.py --label sft --adapter checkpoints/sft-adapter --eval-file data/val.jsonl --output eval_results/sft
!python training/evaluate.py --label dpo --adapter checkpoints/dpo-adapter --eval-file data/val.jsonl --output eval_results/dpo

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 290/290 [00:00<00:00, 870.19it/s]
{
  "label": "base",
  "n_examples": 30,
  "wall_clock_seconds": 113.40273308753967,
  "seconds_per_example": 3.780091102917989,
  "valid_json_rate": 1.0,
  "has_all_fields_rate": 1.0,
  "category_accuracy": 0.0,
  "priority_accuracy": 0.06666666666666667,
  "exact_match_rate": 0.0
}
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao

In [16]:
!python training/compare_results.py eval_results/base.summary.json eval_results/sft.summary.json eval_results/dpo.summary.json

| metric | base | sft | dpo |
|---|---|---|---|
| valid_json_rate | 1.000 | 1.000 | 1.000 |
| has_all_fields_rate | 1.000 | 1.000 | 1.000 |
| category_accuracy | 0.000 | 1.000 | 0.933 |
| priority_accuracy | 0.067 | 0.600 | 0.600 |
| exact_match_rate | 0.000 | 0.033 | 0.000 |
| seconds_per_example | 3.780 | 5.143 | 5.027 |


## 7. Hardware budget summary

Pull the peak-memory/wall-clock numbers from every run into one place for the write-up to Elena and the platform team.

In [17]:
sft_metrics = json.load(open("checkpoints/sft-adapter/run_metrics.json"))
dpo_metrics = json.load(open("checkpoints/dpo-adapter/run_metrics.json"))

print(f"SFT   — {sft_metrics['wall_clock_seconds']:.1f}s, peak GPU memory {sft_metrics['peak_gpu_memory_gb']:.2f} GB")
print(f"DPO   — {dpo_metrics['wall_clock_seconds']:.1f}s, peak GPU memory {dpo_metrics['peak_gpu_memory_gb']:.2f} GB")
print("Free-tier Colab T4 has 16 GB VRAM.")

SFT   — 286.8s, peak GPU memory 1.21 GB
DPO   — 241.3s, peak GPU memory 5.87 GB
Free-tier Colab T4 has 16 GB VRAM.


## 8. Save artifacts

Colab runtimes are ephemeral — zip the adapters and eval results so they survive the session (download, or copy to Drive).

In [18]:
!zip -qr artifacts.zip checkpoints/sft-adapter checkpoints/dpo-adapter eval_results
from google.colab import files
files.download("artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>